# Practice Lab: Agent Architectures (Lesson 11.2)

Module 11 · 8 exercises · Patterns + routing + chaining + fan-out + LangGraph + handoffs + India

Exercises 1-4 run locally (pure Python).


# Lesson 11.2: Agent Architectures — ReAct, Plan, Reflect

8 exercises: 2E + 3M + 3C

Exercises 1-4 run **locally** (pure Python). Ex 5-8 are design.


In [ ]:
import json
import time
import numpy as np
from collections import Counter
np.random.seed(42)


---
## Exercise 1 (Easy): Pattern Matcher


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
tasks = [
    ("Translate email Hindi->English","Augmented LLM","Single call",False),
    ("PDF extract->validate->store","Prompt Chaining","Fixed 3-step",False),
    ("Route tickets billing/tech/returns","Routing","Known categories",False),
    ("Summarize 5 articles independently","Parallelization","Independent tasks",False),
    ("3 LLM opinions on contract","Parallelization (voting)","Majority confidence",False),
    ("Research across unknown sources","Orchestrator-Workers","Dynamic subtasks",True),
    ("Write + refine blog iteratively","Evaluator-Optimizer","Generate+critique loop",True),
    ("Debug failing test suite","Full Agent (ReAct)","Open-ended exploration",True),
    ("Plan multi-city trip","Full Agent (Plan-Execute)","Complex multi-step",True),
    ("Generate report, check, fix","Full Agent (Reflexion)","Self-critique",True),
]

print("Pattern Matcher:")
print(f"  {'#':>2} {'Task':<40} {'Pattern':<24} {'Agent?':>6}")
print(f"  {'-'*75}")
for i,(task,pat,reason,agent) in enumerate(tasks,1):
    print(f"  {i:>2} {task:<40} {pat:<24} {'YES' if agent else 'no':>6}")

agents = sum(1 for t in tasks if t[3])
print(f"\n  Agent needed: {agents}/10. Non-agent handles {10-agents}/10!")
print(f"  Order: Single LLM -> Chaining -> Routing -> Parallel -> Orchestrator -> Agent")


</details>


---
## Exercise 2 (Easy): Routing Agent


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
class Handler:
    def __init__(self,name,model,tools,cost):
        self.name=name;self.model=model;self.tools=tools;self.cost=cost
    def handle(self,q):
        for tn,fn in self.tools.items():
            if any(k in q.lower() for k in tn.split("_")): return {"handler":self.name,"model":self.model,"result":fn(q),"cost":self.cost}
        return {"handler":self.name,"model":self.model,"result":"handled","cost":self.cost}

class Router:
    def __init__(self): self.handlers={}
    def add(self,cat,kws,handler): self.handlers[cat]={"kws":kws,"h":handler}
    def route(self,q):
        scores = {c:sum(1 for k in cfg["kws"] if k in q.lower()) for c,cfg in self.handlers.items()}
        best = max(scores,key=scores.get)
        return self.handlers[best if scores[best]>0 else list(self.handlers.keys())[0]]["h"].handle(q)

router = Router()
router.add("billing",["refund","payment","invoice","bill","price","cost","emi"],
    Handler("billing","haiku-3.5",{"check_refund":lambda q:{"eligible":True}},0.001))
router.add("technical",["error","bug","install","setup","code","not working","crash"],
    Handler("technical","sonnet-4",{"search_docs":lambda q:{"docs":"setup guide"}},0.01))
router.add("courses",["course","enroll","learn","module","genai","python","certificate","schedule"],
    Handler("courses","flash-2.5",{"search_courses":lambda q:{"top":"GenAI"}},0.003))

queries = [("I want a refund","billing"),("Code won't run","technical"),("GenAI courses?","courses"),
           ("Python course cost?","billing"),("Installation failing","technical"),("Next live class?","courses")]

print("Routing Agent:")
correct=0; total_cost=0
for q,exp in queries:
    r = router.route(q); ok = r["handler"]==exp; correct += ok; total_cost += r["cost"]
    print(f"  [{'OK' if ok else 'XX'}] '{q[:30]}' -> {r['handler']} ({r['model']}) ${r['cost']:.3f}")

single_cost = len(queries)*0.01
print(f"\n  Accuracy: {correct}/{len(queries)}={correct/len(queries):.0%}")
print(f"  Cost: ${total_cost:.3f} routed vs ${single_cost:.3f} single = {(1-total_cost/single_cost)*100:.0f}% savings")


</details>


---
## Exercise 3 (Medium): Prompt Chaining Pipeline


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import json

class Chain:
    def __init__(self): self.steps=[]; self.trace=[]
    def add(self,name,proc,validate=None): self.steps.append({"name":name,"proc":proc,"val":validate})
    def run(self,data):
        self.trace=[]
        for s in self.steps:
            data = s["proc"](data); self.trace.append({"step":s["name"],"out":str(data)[:60]})
            if s["val"]:
                ok,msg = s["val"](data)
                if not ok: return {"status":"gate_failed","at":s["name"],"reason":msg,"trace":self.trace}
        return {"status":"complete","result":data,"trace":self.trace}

chain = Chain()
chain.add("extract",
    lambda d:{"name":d.get("raw_name","").strip(),"email":d.get("raw_email","").strip(),"course":d.get("raw_course","").strip()},
    lambda d:(bool(d.get("name")),"Missing name" if not d.get("name") else "OK"))
chain.add("validate",
    lambda d:{**d,"email_ok":"@" in d.get("email",""),"course_ok":d.get("course","").lower() in ["genai","python","cloud"]},
    lambda d:(d.get("email_ok") and d.get("course_ok"),"Invalid email" if not d.get("email_ok") else "Unknown course"))
chain.add("enrich",lambda d:{**d,"price":{"genai":14999,"python":9999,"cloud":11999}.get(d["course"].lower(),0)})
chain.add("format",lambda d:{"student":d["name"],"email":d["email"],"course":d["course"],"total":round(d["price"]*1.18)})

print("Prompt Chaining Pipeline:")
for label,data in [("Valid",{"raw_name":"Priya","raw_email":"p@x.com","raw_course":"GenAI"}),
                    ("Bad email",{"raw_name":"Rahul","raw_email":"bad","raw_course":"python"}),
                    ("No name",{"raw_name":"","raw_email":"x@y.com","raw_course":"cloud"})]:
    r = chain.run(data)
    print(f"  {label}: {r['status']}" + (f" at={r.get('at')} reason={r.get('reason')}" if r['status']!='complete' else f" -> {json.dumps(r['result'])[:60]}"))

print(f"\n  Chain ~95% accuracy (simpler steps) vs single-call ~78%")


</details>


---
## Exercise 4 (Medium): Parallel Fan-Out


In [ ]:
# YOUR CODE
pass


<details><summary>Solution</summary>


In [ ]:
import time
from collections import Counter

def src_web(q): time.sleep(0.3); return {"source":"web","answer":"GenAI: 14999","conf":0.85}
def src_docs(q): time.sleep(0.2); return {"source":"docs","answer":"GenAI: 14999","conf":0.95}
def src_db(q): time.sleep(0.4); return {"source":"db","answer":"GenAI: 14999","conf":0.99}

def aggregate(results):
    answers = {}
    for r in results:
        a = r["answer"]
        if a not in answers: answers[a]={"n":0,"conf":0,"src":[]}
        answers[a]["n"]+=1; answers[a]["conf"]+=r["conf"]; answers[a]["src"].append(r["source"])
    best = max(answers.items(),key=lambda x:x[1]["conf"])
    return {"answer":best[0],"agreement":best[1]["n"],"avg_conf":best[1]["conf"]/best[1]["n"],"sources":best[1]["src"]}

print("Parallel Fan-Out:")
start=time.time()
results=[src_web("genai"),src_docs("genai"),src_db("genai")]
seq_ms=(time.time()-start)*1000
par_ms=max(300,200,400)

print(f"  Sequential: {seq_ms:.0f}ms")
print(f"  Parallel:   {par_ms}ms (limited by slowest)")
print(f"  Speedup:    {seq_ms/par_ms:.1f}x")

agg = aggregate(results)
print(f"\n  Aggregation: {agg['answer']}, {agg['agreement']}/3 agree, {agg['avg_conf']:.0%} conf")

votes = ["14999","14999","15000"]
maj = Counter(votes).most_common(1)[0]
print(f"  Voting: '{maj[0]}' wins {maj[1]}/3 ({maj[1]/len(votes):.0%} confidence)")
print(f"\n  Fan-out: LangGraph 137x speedup. Aggregation = hard problem")


</details>


---
## Exercise 5 (Medium): LangGraph State + Checkpoints
Design/architecture. See practice-lab-11_2.html.


In [ ]:
# Exercise 5: LangGraph State + Checkpoints
pass


<details><summary>Solution</summary>


In [ ]:
print('LangGraph: StateGraph + TypedDict + Annotated reducers')
print('add_messages=smart append, operator.add=concat, no annotation=overwrite')
print('Checkpoint: PostgresSaver, auto-saves at super-step')
print('Time-travel: get_state_history -> update_state -> invoke(None, fork)')
print('HITL: interrupt() pauses, Command(resume=val) resumes')


</details>


---
## Exercise 6 (Challenge): Supervisor-Worker System
Design/architecture. See practice-lab-11_2.html.


In [ ]:
# Exercise 6: Supervisor-Worker System
pass


<details><summary>Solution</summary>


In [ ]:
print('Supervisor + 3 workers: overhead = supervisor call tokens')
print('Single: 2000 tok. Multi: 2800 tok (+40%)')
print('Reliability: single 95% vs 3-chain 85.7%')
print('Google: 39-70% DEGRADATION on sequential tasks')
print('Use only for genuinely parallel + different skills')


</details>


---
## Exercise 7 (Challenge): OpenAI SDK Handoff Chain
Design/architecture. See practice-lab-11_2.html.


In [ ]:
# Exercise 7: OpenAI SDK Handoff Chain
pass


<details><summary>Solution</summary>


In [ ]:
print('OpenAI SDK: handoff() creates transfer_to_agent tools')
print('Context: full history (default) or input_filter(keep_last_n)')
print('Loop prevention: max_handoffs=3, visited set, one-way')
print('Claude: clean-slate sub-agents (depth=1, ~10 concurrent)')
print('Handoff=conversation routing. Clean-slate=parallel tasks')


</details>


---
## Exercise 8 (Challenge): India Voice Pipeline
Design/architecture. See practice-lab-11_2.html.


In [ ]:
# Exercise 8: India Voice Pipeline
pass


<details><summary>Solution</summary>


In [ ]:
print('Sarvam Pipeline: STT(30rs/hr) -> LLM(FREE!) -> Translate -> TTS(30rs/10K)')
print('Per 1000 calls: ~2640 rs vs $113-148 global (80% cheaper)')
print('India Stack: Aadhaar $0.15, UPI 10B txns, DigiLocker 676M users')
print('API Setu: 6400+ govt APIs as agent tools')


</details>
